In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report,accuracy_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

In [2]:
df=pd.read_csv('train_data.txt',sep=':::',names=['ID','TITLE','GENRE','DESCRIPTION'])

C:\Users\ASUS\AppData\Local\Temp\ipykernel_16812\305693459.py:1: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  df=pd.read_csv('train_data.txt',sep=':::',names=['ID','TITLE','GENRE','DESCRIPTION'])


In [3]:
df.head()

,ID,TITLE,GENRE,DESCRIPTION
0,1,Oscar et la dame rose (2009),drama,Listening in to a conversation between his do...
1,2,Cupid (1997),thriller,A brother and sister with a past incestuous r...
2,3,"Young, Wild and Wonderful (1980)",adult,As the bus empties the students for their fie...
3,4,The Secret Sin (1915),drama,To help their unemployed father make ends mee...
4,5,The Unrecovered (2007),drama,The film's title refers not only to the un-re...


In [4]:
df['DESCRIPTION_CLEAN'] = df['DESCRIPTION'].str.lower().str.replace('[^\w\s]', '', regex=True)
#this will remove all the punctuation from the description column

<>:1: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
<>:1: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
C:\Users\ASUS\AppData\Local\Temp\ipykernel_16812\895022045.py:1: SyntaxWarning: "\w" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\w"? A raw string is also an option.
  df['DESCRIPTION_CLEAN'] = df['DESCRIPTION'].str.lower().str.replace('[^\w\s]', '', regex=True)


In [5]:
df.head()

,ID,TITLE,GENRE,DESCRIPTION,DESCRIPTION_CLEAN
0,1,Oscar et la dame rose (2009),drama,Listening in to a conversation between his do...,listening in to a conversation between his do...
1,2,Cupid (1997),thriller,A brother and sister with a past incestuous r...,a brother and sister with a past incestuous r...
2,3,"Young, Wild and Wonderful (1980)",adult,As the bus empties the students for their fie...,as the bus empties the students for their fie...
3,4,The Secret Sin (1915),drama,To help their unemployed father make ends mee...,to help their unemployed father make ends mee...
4,5,The Unrecovered (2007),drama,The film's title refers not only to the un-re...,the films title refers not only to the unreco...


In [6]:
X=df['DESCRIPTION_CLEAN']
y=df['GENRE']

In [7]:
Xtrain,Xtest,ytrain,ytest=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [8]:
Xtrain.shape

(43371,)

In [9]:
tf = TfidfVectorizer(stop_words='english')
Xtraintf=tf.fit_transform(Xtrain)
Xtesttf=tf.transform(Xtest)


In [10]:
Xtraintf.shape


(43371, 132045)

In [11]:
Xtesttf.shape

(10843, 132045)

In [ ]:
models = {
    "SVM": LinearSVC(random_state=42,class_weight='balanced'),
    "Multinomial NB": MultinomialNB(fit_prior=False),
    "Logistic Regression": LogisticRegression(class_weight='balanced')
}

results = {}
for name, model in models.items():
    model.fit(Xtraintf, ytrain)
    ypred = model.predict(Xtesttf)
    acc = accuracy_score(ytest, ypred)
    results[name] = round(acc * 100, 2)
    print(f"\n{name} — Accuracy: {acc*100:.2f}%")
    print(classification_report(ytest, ypred))

best = max(results, key=results.get)
print(f"\nBest model: {best} ({results[best]}%)")

In [14]:
import joblib

joblib.dump(models["SVM"], 'svm_model.pkl')
joblib.dump(tf, 'tfidf_vectorizer.pkl')

['tfidf_vectorizer.pkl']

In [15]:
test_load = joblib.load('svm_model.pkl')
test_load.predict(Xtesttf[:5])

array([' short ', ' horror ', ' adult ', ' drama ', ' comedy '],
      dtype=object)